# 10장 보안 실습 — Journal·Audit 이벤트 연결


## Goal

서비스의 부팅 문맥과 Audit 레코드·이벤트 단위를 구분합니다.

[교안과 분석 질문](../../10-program-architecture/10-4-audit-analysis.md)을 먼저 읽습니다.


## Setup

Python 커널의 %%bash를 사용합니다. 새 임시 폴더에 합성 자료와 결과 경로를 준비합니다. 외부 접속·서비스 등록·원본 서버 조사는 하지 않습니다. 코드를 검토하고 Setup부터 순서대로 실행합니다. Bash 셀 사이의 상태는 환경 변수와 파일로 전달합니다.


In [ ]:
from pathlib import Path
import hashlib
import os
import tempfile

lab = Path(tempfile.mkdtemp(prefix='bash-security-10-'))
data = lab / 'data'
output = lab / 'output'
data.mkdir()
output.mkdir()
fixtures = {'journal-review.psv': 'time_kst|boot_id|unit|pid|message\n2026-09-10T08:00:00+09:00|BOOT-A|init.scope|1|System boot\n2026-09-10T09:02:00+09:00|BOOT-A|ssh.service|104|Accepted publickey for analyst\n2026-09-10T09:05:00+09:00|BOOT-A|report-helper.service|1|Started report helper\n2026-09-10T09:06:00+09:00|BOOT-A|report-helper.service|520|Report completed\n', 'audit.log': 'type=SYSCALL msg=audit(1788998580.000:900): arch=c000003e syscall=59 success=yes exit=0 a0=0 a1=0 a2=0 a3=0 items=1 ppid=410 pid=450 auid=1000 uid=0 gid=0 euid=0 suid=0 fsuid=0 egid=0 sgid=0 fsgid=0 tty=pts0 ses=4 comm="id" exe="/usr/bin/id" key="course_exec"\ntype=EXECVE msg=audit(1788998580.000:900): argc=1 a0="id"\ntype=CWD msg=audit(1788998580.000:900): cwd="/home/analyst"\ntype=PATH msg=audit(1788998580.000:900): item=0 name="/usr/bin/id" inode=100 dev=08:01 mode=0100755 ouid=0 ogid=0 rdev=00:00 nametype=NORMAL\ntype=PROCTITLE msg=audit(1788998580.000:900): proctitle=6964\ntype=EOE msg=audit(1788998580.000:900):\n'}
for name, content in fixtures.items():
    path = data / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
before = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
tools_dir = lab / 'tools'
tools_dir.mkdir()
os.environ['COURSE_TOOLS'] = str(tools_dir)
os.environ['COURSE_DATA'] = str(data)
os.environ['COURSE_OUT'] = str(output)
print('합성 자료와 새 결과 폴더 준비 완료')


## Steps

예상 결과: 서비스 2행, Audit 6레코드/1이벤트, auid1000/euid0은 승인 여부와 별개

명령을 실행하기 전에 입력·출력·실패 조건을 표시합니다. 자료의 상세 필드 해석과 정상 행위 대안은 연결된 교안에서 확인합니다.


### 1. Journal 요약에서 서비스와 부팅 문맥 선택


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
awk -F '|' 'NR>1 && $3=="report-helper.service" {print $1 "|" $2 "|" $5}' \
 "$COURSE_DATA/journal-review.psv" > "$COURSE_OUT/service-events.psv"
test "$(wc -l < "$COURSE_OUT/service-events.psv")" -eq 2
cat "$COURSE_OUT/service-events.psv"


### 2. Audit 레코드와 이벤트 수 구분


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
sed -n 's/.*msg=audit(\([^)]*\)).*/\1/p' "$COURSE_DATA/audit.log" | LC_ALL=C sort -u > "$COURSE_OUT/event-ids.txt"
test "$(wc -l < "$COURSE_DATA/audit.log")" -eq 6
test "$(wc -l < "$COURSE_OUT/event-ids.txt")" -eq 1
grep -Fx '1788998580.000:900' "$COURSE_OUT/event-ids.txt"
printf 'audit_records=6 audit_events=1\n'


### 3. 원래 로그인 ID와 실행 권한 구분


In [ ]:
%%bash
set -euo pipefail
: "${COURSE_DATA:?}" "${COURSE_OUT:?}"
grep -F 'type=SYSCALL ' "$COURSE_DATA/audit.log" > "$COURSE_OUT/syscall.txt"
grep -qF 'auid=1000 uid=0 gid=0 euid=0' "$COURSE_OUT/syscall.txt"
grep -qF 'exe="/usr/bin/id"' "$COURSE_OUT/syscall.txt"
printf 'auid=1000 euid=0 command=id approval=not_in_audit_record\n'


## Checks

각 STEP의 test는 고정 자료의 계산 결과를 검사합니다. 아래는 원본 내용 보존을 확인합니다. 실행 성공과 침해 판정은 다릅니다. 어떤 결과가 사실이고 어떤 결론이 가설인지 교안 질문에 답합니다.


In [ ]:
after = {name: hashlib.sha256((data / name).read_bytes()).hexdigest() for name in fixtures}
assert before == after
print('원본 내용 보존: PASS')
print('분석 결과 파일 수:', sum(p.is_file() for p in output.rglob('*')))


## Next Steps

교안의 완료 기준에 따라 근거·정상 행위 가능성·누락·추가 확인을 제출합니다. 결과는 검토용 임시 폴더에 남습니다. 재실행은 Setup부터 새 폴더에서 시작하며 실제 증거를 공개 저장소에 올리지 않습니다.
